In [1]:
import os
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List
from langchain.schema import Document
from langchain.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

c:\Users\user\.conda\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# os.chdir("../")
# os.chdir("MEDICAL_CHATBOT")

In [11]:
%pwd

'd:\\My Files\\New Projects\\Medical Chatbot\\MEDICAL_CHATBOT'

In [7]:
## Function to load text from pdf

def load_pdf_files(pdf_path):
    loader = DirectoryLoader(pdf_path, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [12]:
extracted_data = load_pdf_files("Data")

In [13]:
len(extracted_data)

637

In [ ]:
extracted_data[10].page_content     

'Rhonda Cloos, R.N.\nMedical Writer\nAustin, TX\nGloria Cooksey, C.N.E\nMedical Writer\nSacramento, CA\nAmy Cooper, M.A., M.S.I.\nMedical Writer\nVermillion, SD\nDavid A. Cramer, M.D.\nMedical Writer\nChicago, IL\nEsther Csapo Rastega, R.N., B.S.N.\nMedical Writer\nHolbrook, MA\nArnold Cua, M.D.\nPhysician\nBrooklyn, NY\nTish Davidson, A.M.\nMedical Writer\nFremont, California\nDominic De Bellis, Ph.D.\nMedical Writer/Editor\nMahopac, NY\nLori De Milto\nMedical Writer\nSicklerville, NJ\nRobert S. Dinsmoor\nMedical Writer\nSouth Hamilton, MA\nStephanie Dionne, B.S.\nMedical Writer\nAnn Arbor, MI\nMartin W. Dodge, Ph.D.\nTechnical Writer/Editor\nCentinela Hospital and Medical\nCenter\nInglewood, CA\nDavid Doermann\nMedical Writer\nSalt Lake City, UT\nStefanie B. N. Dugan, M.S.\nGenetic Counselor\nMilwaukee, WI\nDoug Dupler, M.A.\nScience Writer\nBoulder, CO\nJulie A. Gelderloos\nBiomedical Writer\nPlaya del Rey, CA\nGary Gilles, M.A.\nMedical Writer\nWauconda, IL\nHarry W. Golden\nMedica

In [19]:
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """Given a list of document objects, return a new list of document objects
    containing only source in metadata and the original page content."""

    return [
        Document(
            page_content=doc.page_content,
            metadata={"source": doc.metadata.get("source")},
        )
        for doc in docs
    ]

In [20]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [23]:
minimal_docs[:5]

[Document(metadata={'source': 'Data\\Medical_book.pdf'}, page_content=''),
 Document(metadata={'source': 'Data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'Data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': 'Data\\Medical_book.pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow,Manager, Imaging and Multimedia\nContent\nRobyn V . Young,Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imaging and\nMultimed

In [ ]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

In [27]:
text_chunks = text_split(minimal_docs)
print(f"Number of text chunks: {len(text_chunks)}")

Number of text chunks: 3426


In [30]:
text_chunks[:5]

[Document(metadata={'source': 'Data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'Data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': 'Data\\Medical_book.pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow,Manager, Imaging and Multimedia\nContent\nRobyn V . Young,Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imaging and\nMultimedia Content\nKelly A. Quin, Editor, Imaging and Multimedia Content\nLeitha E

In [7]:
def download_embeddings():
    """
    Download and return the Hugging Face embeddings."""
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    # text_embeddings = embeddings.embed_documents([chunk.page_content for chunk in text_chunks])
    return embeddings

In [8]:
embeddings = download_embeddings()

C:\Users\user\AppData\Local\Temp\ipykernel_8560\788476943.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [9]:
vector = embeddings.embed_query("Hello World")
print(vector)

[-0.03447720408439636, 0.031023239716887474, 0.00673496862873435, 0.026108969002962112, -0.03936196118593216, -0.16030246019363403, 0.06692393124103546, -0.0064414795488119125, -0.047450557351112366, 0.014758911915123463, 0.0708753690123558, 0.05552756413817406, 0.01919337548315525, -0.026251327246427536, -0.010109500028192997, -0.026940541341900826, 0.022307470440864563, -0.02222665585577488, -0.14969269931316376, -0.01749308407306671, 0.007676247972995043, 0.054352279752492905, 0.003254473675042391, 0.03172597661614418, -0.0846213549375534, -0.0294059906154871, 0.051595624536275864, 0.048124030232429504, -0.003314792178571224, -0.05827920511364937, 0.04196930304169655, 0.022210685536265373, 0.1281888484954834, -0.02233896590769291, -0.011656301096081734, 0.06292833387851715, -0.032876282930374146, -0.09122605621814728, -0.031175389885902405, 0.05269956216216087, 0.0470348559319973, -0.08420302718877792, -0.03005620837211609, -0.02074478194117546, 0.009517811238765717, -0.003721802262

In [15]:
print(f"Length of the vector: {len(vector)}")

Length of the vector: 384


In [2]:
load_dotenv()

True

In [3]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [5]:
# print(PINECONE_API_KEY)

In [ ]:
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [13]:
pc = Pinecone(api_key=PINECONE_API_KEY)

In [47]:
pc

In [14]:
index_name = "medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(name=index_name,
    dimension=384, # Dimension of the embedding vector
    metric="cosine", 
    spec=ServerlessSpec(cloud="aws", region="us-east-1"))

index = pc.Index(index_name)

In [ ]:
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [15]:
## Load Existing Index

# Embed each chunk and add it to the Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name=index_name
)

In [ ]:
## Add more data to the existing index
# dummy_data = Document(
#         page_content="This is a dummy document for testing purposes."
        # metadata={"source": "dummy_source.pdf"}
#     )
#
# docsearch.add_documents(dummy_data)

In [16]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwargs={"k": 3})

In [17]:
retrieved_docs = retriever.invoke("What are the symptoms of diabetes?")
retrieved_docs

[Document(id='8cecb03b-e11d-4331-9805-707f7da66834', metadata={'source': 'Data\\Medical_book.pdf'}, page_content='begin to fall. A person with diabetes mellitus either does\nnot make enough insulin, or makes insulin that does not\nwork properly. The result is blood sugar that remains\nhigh, a condition called hyperglycemia.\nDiabetes must be diagnosed as early as possible. If\nleft untreated, it can damage or cause failure of the eyes,\nkidneys, nerves, heart, blood vessels, and other body\norgans. Hypoglycemia, or low blood sugar, may also be\ndiscovered through blood sugar testing. Hypoglycemia is\ncaused by various hormone disorders and liver disease,\nas well as by too much insulin.\nDescription\nThere are a variety of ways to measure a person’s\nblood sugar.\nWhole blood glucose test\nWhole blood glucose testing can be performed by a\nperson in his or her home, and kits are available for this\npurpose. The person pricks his or her finger (a finger\nstick) with a sterile sharp blad

In [22]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [19]:
system_prompt = (
    "You are a helpful medical assistant for question answering tasks related to the medical domain. "
    "Use the following retrieved documents to answer the question as accurately as possible. "
    "If you don't know the answer, say that you don't know. "
    "Use 3 sentences maximum and keep the answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [24]:
question_answering_chain = create_stuff_documents_chain(llm,prompt)
rag_chain = create_retrieval_chain(retriever, question_answering_chain)

In [25]:
response = rag_chain.invoke({"input": "What are the symptoms of diabetes?"})
print(response["answer"])

A person with diabetes mellitus may experience symptoms such as increased thirst and urination, blurred vision, and fatigue due to high blood sugar levels. If left untreated, diabetes can cause damage to various body organs. Additionally, hypoglycemia, or low blood sugar, may also be discovered through blood sugar testing.


In [26]:
response2 = rag_chain.invoke({"input": "How to lose weight?"})
print(response2["answer"])

To lose weight, it is recommended to maintain a desirable body weight by eating right and exercising regularly. This can help reduce total and LDL cholesterol, reduce triglycerides, and boost HDL cholesterol, which may also reduce blood pressure. Eating a healthy diet and engaging in aerobic exercise, such as walking or cycling, can also aid in weight loss.
